# Wire 12 quality assertions with two severities: warnings page on-call, critical blocks the gold write

The CFO's Monday board meeting reads from your gold layer. One bad gold load and you spend Tuesday explaining the numbers. Today you build the wall that stops the bad load before it lands. Twelve assertions you actually believe in. Two severities you can defend in a post-mortem. One deliberate violation that proves the wiring before you ship.

The stance this lab takes:
- Row-count drops over 30% are critical. The load is wrong, do not push it.
- Schema and uniqueness violations are critical. If `event_id` is not unique, downstream joins lie.
- Freshness slips are a warning. The data is stale, but pushing stale numbers to the CFO is better than pushing the wrong customer aggregate.

You will build the framework end to end:
1. Glue Data Quality ruleset with 12 DQDL assertions across six families.
2. A router Lambda that takes the DQ evaluation result and tags the run as `clean`, `warning`, or `critical`.
3. A Step Functions state machine that runs the DQ evaluation, calls the router, branches on severity, and either blocks the gold write (critical), publishes to SNS while still writing gold (warning), or writes gold directly (clean).
4. Three executions that prove the wiring: one clean, one warning, one critical.

**Time.** Plan for about 75 minutes hands-on. If you hit Glue DQ propagation snags or ASL debugging, budget up to 90 minutes. Cleanup is the final cell and brings the AWS bill back to zero.

**Cost.** Realistic session: $0.40 to $1.00. The Glue DQ evaluations themselves cost pennies (about 0.1 cent per evaluation at this row count). The Athena Iceberg writes are the largest line item. The critical run never writes gold, which is the whole point of the lab.

This lab maps to the Data Engineering job-prep track, Sub-track D: Data Reliability Engineering. Differentiation: DEA-C01 does not cover Glue Data Quality, and the Databricks DLT-expectations lab covers the same idea on a different stack. The AWS-native pattern in this lab plus the severity routing layer is what the interview talking point looks like.

In [ ]:
# NBVAL_SKIP
# Install labrun-checks. Pinned to a specific version per LAB-CREATION-BLUEPRINT.md
# build rules. Never use unpinned installs.

!pip install --quiet labrun-checks==0.7.0

In [ ]:
# NBVAL_SKIP
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern from LAB-CREATION-BLUEPRINT.md
# section 12.

import atexit
import getpass
import io
import json
import re
import time
import zipfile
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CheckpointResult,
    CleanupResource,
)

LAB_ID = "data-quality-framework"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID  # must equal the cert/track YAML slug exactly
REGION = "us-east-1"  # all DE-track labs run in us-east-1 per LAB-CREATION-BLUEPRINT section 15

# Resource name constants. Bucket gets resolved after STS GetCallerIdentity
# returns the account ID.
BUCKET_NAME = None
DB = f"labrun-{LAB_ID}-db"
RULESET_NAME = f"labrun-{LAB_ID}-ruleset"
DQ_ROLE = f"labrun-{LAB_ID}-role"
ROUTER_NAME = f"labrun-{LAB_ID}-router"
SM_NAME = f"labrun-{LAB_ID}-sm"
RULE_NAME = f"labrun-{LAB_ID}-rule"
TOPIC_NAME = f"labrun-{LAB_ID}-alerts"
INLINE_POLICY_NAME = f"labrun-{LAB_ID}-inline"

# Resolved later after AWS calls return real ARNs.
ROLE_ARN = None
ROUTER_ARN = None
TOPIC_ARN = None
STATE_MACHINE_ARN = None
ATHENA_OUTPUT_LOC = None

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# and confirm the region. This cell must succeed before the manifest cell
# creates anything per LAB-CREATION-BLUEPRINT section 15.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"This lab runs in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

# Validate credentials against AWS via STS GetCallerIdentity. Fail fast with a
# clear message rather than waiting for the first IAM call to error out.
sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")
print(f"Session token in use: {bool(aws_session_token)}")

# Resolve names that depend on account_id.
BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{DQ_ROLE}"
ROUTER_ARN = f"arn:aws:lambda:{REGION}:{ACCOUNT_ID}:function:{ROUTER_NAME}"
TOPIC_ARN = f"arn:aws:sns:{REGION}:{ACCOUNT_ID}:{TOPIC_NAME}"
STATE_MACHINE_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{SM_NAME}"
ATHENA_OUTPUT_LOC = f"s3://{BUCKET_NAME}/athena-results/"
print(f"Bucket name resolved: {BUCKET_NAME}")

# Register the session with labrun-checks. register() returns None;
# do not assign its return value.
register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# Cleanup manifest, atexit safety net, and orphan scan.
# Manifest is module-level and in strict reverse-creation order. No critical
# (hourly-billed) resources: Glue DQ bills per evaluation, Step Functions
# Standard bills per transition, Lambda free at lab volume, Athena bills per
# TB scanned (our datasets scan KB), SNS free at lab volume.
#
# Per RESOURCE-SAFETY-SPEC Rule 4, the orphan scan blocks execution if any
# tagged resources from a prior session exist (not just print a warning).
#
# The default AwsCleanupAdapter does not ship handlers for sfn_state_machine
# or glue_dq_ruleset (and treats iceberg tables as plain glue_table). The
# inline _LabAdapter at the bottom of this notebook adds those three types.

CLEANUP_MANIFEST = [
    CleanupResource(
        type="sfn_state_machine",
        id=STATE_MACHINE_ARN,
        region=REGION,
        cli_delete_command=f"aws stepfunctions delete-state-machine --state-machine-arn {STATE_MACHINE_ARN}",
    ),
    CleanupResource(
        type="eventbridge_rule",
        id=RULE_NAME,
        region=REGION,
        cli_delete_command=(
            f"aws events remove-targets --rule {RULE_NAME} --ids 1 && "
            f"aws events delete-rule --name {RULE_NAME}"
        ),
    ),
    CleanupResource(
        type="lambda_function",
        id=ROUTER_NAME,
        region=REGION,
        cli_delete_command=f"aws lambda delete-function --function-name {ROUTER_NAME}",
    ),
    CleanupResource(
        type="sns_topic",
        id=TOPIC_NAME,
        region=REGION,
        cli_delete_command=f"aws sns delete-topic --topic-arn {TOPIC_ARN}",
    ),
    CleanupResource(
        type="glue_dq_ruleset",
        id=RULESET_NAME,
        region=REGION,
        cli_delete_command=f"aws glue delete-data-quality-ruleset --name {RULESET_NAME}",
    ),
    CleanupResource(
        type="iceberg_table",
        id="events_gold",
        parent=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB} --name events_gold",
        extra={"location": f"s3://{BUCKET_NAME}/events_gold/"},
    ),
    CleanupResource(
        type="iceberg_table",
        id="events_silver",
        parent=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB} --name events_silver",
        extra={"location": f"s3://{BUCKET_NAME}/events_silver/"},
    ),
    CleanupResource(
        type="iceberg_table",
        id="ref_customers",
        parent=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-table --database-name {DB} --name ref_customers",
        extra={"location": f"s3://{BUCKET_NAME}/ref_customers/"},
    ),
    CleanupResource(
        type="iam_role",
        id=DQ_ROLE,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {DQ_ROLE}",
    ),
    CleanupResource(
        type="glue_database",
        id=DB,
        region=REGION,
        cli_delete_command=f"aws glue delete-database --name {DB}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    """Best-effort teardown on kernel shutdown.

    The dedicated cleanup cell is the authoritative entry point; this is
    the safety net for kernel crashes. Errors are swallowed because atexit
    handlers must not raise.
    """
    try:
        if "_LabAdapter" in globals():
            run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())
        else:
            run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list[str]:
    """Refuse to start if a previous run left tagged resources behind.

    Per RESOURCE-SAFETY-SPEC Rule 4, the setup cell must check for
    leftover state with the lab's tag and surface the cleanup command
    before creating any new resources.
    """
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns: list[str] = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("These are leftovers from a previous run of this lab. Re-running")
    print("setup against an unclean state can produce duplicate resources or")
    print("collide on the role and state-machine names. Run the cleanup cell")
    print("at the bottom of this notebook first, then re-run setup.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Stand up the foundation, seed clean silver, and register the 12-assertion DQ ruleset

Five things to land in this task before the routing layer matters:

1. The S3 bucket and Glue database, both tagged with `labrun:lab-slug=data-quality-framework` so the cleanup tag audit can find them.
2. The DQ + Lambda + Step Functions service role. One IAM role with a multi-principal trust policy (Glue, Lambda, Step Functions). The inline policy grants S3 on the lab bucket, Glue DQ + table operations, Athena `StartQueryExecution`, SNS Publish on the lab topic, Lambda Invoke on the router function, CloudWatch Logs, and `iam:PassRole` on itself so Glue DQ can run under this role.
3. Three Iceberg tables in the Glue catalog: `events_silver`, `events_gold`, and `ref_customers`. Iceberg is the right call here because Glue DQ supports it natively and you get DELETE + INSERT for cheap dataset swaps.
4. Seed `ref_customers` with five valid customer IDs (used by the ReferentialIntegrity rule) and `events_silver` with 40 rows of fresh, valid events.
5. The DQ ruleset itself. Twelve rules across six families: ColumnExists (4), Uniqueness (1), ColumnValues (2), ReferentialIntegrity (1), RowCount (2), Freshness (2). The ruleset targets `events_silver`.

DQDL is Glue's quality DSL. It is plain-text rules; the severity is NOT a DQDL concept. Severity is a routing-layer decision that lives in the router Lambda you build in Task 2.

The cell ends with a 15-second IAM propagation wait. Glue DQ runs that try to use the role before propagation lands fail with a confusing assume-role error.

In [ ]:
# NBVAL_SKIP
# Task 1: Bucket, Glue database, IAM role, three Iceberg tables, seed clean
# silver, register the 12-rule DQ ruleset.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
glue = boto3.client(
    "glue",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Bucket. us-east-1 rejects LocationConstraint; other regions require it.
try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    s3.put_bucket_tagging(
        Bucket=BUCKET_NAME,
        Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
    )
    print(f"Bucket created and tagged: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists, reusing.")
    else:
        raise

# Glue database.
try:
    glue.create_database(
        DatabaseInput={"Name": DB, "Description": f"DQ framework catalog for {LAB_ID}"},
    )
    glue.tag_resource(
        ResourceArn=f"arn:aws:glue:{REGION}:{ACCOUNT_ID}:database/{DB}",
        TagsToAdd={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Glue database created and tagged: {DB}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        print(f"Glue database {DB} already exists, reusing.")
    else:
        raise

# IAM role. Multi-principal trust covers Glue (DQ runs), Lambda (router
# execution), and States (Step Functions execution).
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": [
                "glue.amazonaws.com",
                "lambda.amazonaws.com",
                "states.amazonaws.com",
            ]},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=DQ_ROLE,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description=f"labrun {LAB_ID} role (Glue DQ, Lambda router, Step Functions)",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Role created: {DQ_ROLE}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {DQ_ROLE} already exists, reusing.")
    else:
        raise

# Inline policy. Single statement-bag covers every service this role touches.
# Lab-scoped to the lab bucket, the lab database, the lab topic, and the
# router function.
inline_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:*"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": [
                "glue:*DataQuality*",
                "glue:GetTable",
                "glue:GetTables",
                "glue:GetDatabase",
                "glue:GetDatabases",
                "glue:GetPartition",
                "glue:GetPartitions",
                "glue:CreateTable",
                "glue:DeleteTable",
                "glue:UpdateTable",
                "glue:BatchCreatePartition",
                "glue:BatchDeletePartition",
                "glue:BatchGetPartition",
                "glue:BatchGetDataQualityResult",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": [
                "athena:StartQueryExecution",
                "athena:GetQueryExecution",
                "athena:GetQueryResults",
                "athena:StopQueryExecution",
                "athena:GetWorkGroup",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": ["sns:Publish"],
            "Resource": TOPIC_ARN,
        },
        {
            "Effect": "Allow",
            "Action": ["lambda:InvokeFunction"],
            "Resource": ROUTER_ARN,
        },
        {
            "Effect": "Allow",
            "Action": [
                "logs:CreateLogGroup",
                "logs:CreateLogStream",
                "logs:PutLogEvents",
            ],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": ["iam:PassRole"],
            "Resource": ROLE_ARN,
        },
    ],
}
iam.put_role_policy(
    RoleName=DQ_ROLE,
    PolicyName=INLINE_POLICY_NAME,
    PolicyDocument=json.dumps(inline_policy),
)
print(f"Inline policy attached: {INLINE_POLICY_NAME}")

# Helper: run an Athena query, poll until terminal, return state and ExecutionId.
def run_athena(sql: str, timeout: int = 90) -> dict:
    resp = athena.start_query_execution(
        QueryString=sql,
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT_LOC},
        QueryExecutionContext={"Database": DB},
    )
    qid = resp["QueryExecutionId"]
    deadline = time.time() + timeout
    while time.time() < deadline:
        st = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        state = st["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            return {"id": qid, "state": state, "reason": st.get("StateChangeReason", "")}
        time.sleep(2)
    return {"id": qid, "state": "TIMEOUT", "reason": "polling deadline elapsed"}


# Create Iceberg tables via Athena DDL. Athena Iceberg requires the engine
# version 3+ workgroup (the default `primary` workgroup defaults to v3 in 2026).
ddl_silver = (
    f"CREATE TABLE IF NOT EXISTS {DB}.events_silver (\n"
    "  event_id string,\n"
    "  customer_id string,\n"
    "  amount double,\n"
    "  event_timestamp timestamp\n"
    ")\n"
    f"LOCATION 's3://{BUCKET_NAME}/events_silver/'\n"
    "TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
ddl_gold = (
    f"CREATE TABLE IF NOT EXISTS {DB}.events_gold (\n"
    "  event_id string,\n"
    "  customer_id string,\n"
    "  amount double,\n"
    "  event_timestamp timestamp\n"
    ")\n"
    f"LOCATION 's3://{BUCKET_NAME}/events_gold/'\n"
    "TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
ddl_ref = (
    f"CREATE TABLE IF NOT EXISTS {DB}.ref_customers (\n"
    "  customer_id string\n"
    ")\n"
    f"LOCATION 's3://{BUCKET_NAME}/ref_customers/'\n"
    "TBLPROPERTIES ('table_type'='ICEBERG', 'format'='parquet')"
)
for label, ddl in [("events_silver", ddl_silver), ("events_gold", ddl_gold), ("ref_customers", ddl_ref)]:
    r = run_athena(ddl)
    if r["state"] != "SUCCEEDED":
        raise RuntimeError(f"DDL for {label} failed: {r}")
    print(f"Iceberg table ready: {label}")

# Seed ref_customers with five valid IDs. The lab's silver dataset uses
# only C001 through C005 for happy-path rows, so ReferentialIntegrity is 1.0.
seed_ref = (
    f"INSERT INTO {DB}.ref_customers (customer_id) VALUES "
    "('C001'), ('C002'), ('C003'), ('C004'), ('C005')"
)
# First DELETE in case this is a re-run; INSERT INTO without DELETE would
# duplicate the seed rows on every kernel restart.
run_athena(f"DELETE FROM {DB}.ref_customers")
r = run_athena(seed_ref)
if r["state"] != "SUCCEEDED":
    raise RuntimeError(f"ref_customers seed failed: {r}")
print("ref_customers seeded with 5 customer IDs.")

# Seed clean silver. 40 rows, all customer IDs in C001..C005, all amounts
# inside [0, 100000], all event_timestamps fresh (use current_timestamp).
def silver_clean_sql() -> str:
    rows = []
    for i in range(1, 41):
        cid = f"C00{((i - 1) % 5) + 1}"
        amt = round(10 + (i * 3.7) % 200, 2)
        # current_timestamp resolves at query time; same for every row in the batch.
        rows.append(f"('E{i:04d}', '{cid}', {amt}, current_timestamp)")
    values = ", ".join(rows)
    return f"INSERT INTO {DB}.events_silver (event_id, customer_id, amount, event_timestamp) VALUES {values}"

run_athena(f"DELETE FROM {DB}.events_silver")
r = run_athena(silver_clean_sql())
if r["state"] != "SUCCEEDED":
    raise RuntimeError(f"clean silver seed failed: {r}")
print("events_silver seeded with 40 clean rows.")

# DQDL ruleset. Twelve rules across six families. The Target is set at
# the API level (TargetTable parameter), not in DQDL.
DQDL = (
    "Rules = [\n"
    "    ColumnExists \"event_id\",\n"
    "    ColumnExists \"customer_id\",\n"
    "    ColumnExists \"amount\",\n"
    "    ColumnExists \"event_timestamp\",\n"
    "    IsUnique \"event_id\",\n"
    "    ColumnValues \"amount\" between 0 and 100000,\n"
    "    ColumnValues \"customer_id\" matches \"C[0-9]+\",\n"
    "    ReferentialIntegrity \"customer_id\" \"ref_customers.customer_id\" >= 0.95,\n"
    "    RowCount > 20,\n"
    "    RowCount between 25 and 200,\n"
    "    DataFreshness \"event_timestamp\" <= 24 hours,\n"
    "    DataFreshness \"event_timestamp\" >= 0 hours\n"
    "]"
)
try:
    glue.create_data_quality_ruleset(
        Name=RULESET_NAME,
        Description="12-rule DQ ruleset for the data-quality-framework lab",
        Ruleset=DQDL,
        TargetTable={"DatabaseName": DB, "TableName": "events_silver"},
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"DQ ruleset registered: {RULESET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "AlreadyExistsException":
        # Update in place so re-runs after a partial cleanup still land the
        # ruleset content the checkpoint expects.
        glue.update_data_quality_ruleset(
            Name=RULESET_NAME,
            Description="12-rule DQ ruleset for the data-quality-framework lab",
            Ruleset=DQDL,
        )
        print(f"DQ ruleset {RULESET_NAME} already existed, updated in place.")
    else:
        raise

# IAM propagation. Glue DQ validates the role at evaluation-run time and the
# state machine validates at execution-start time. 15 seconds covers nearly
# all observed propagation cases in us-east-1.
print("Your IAM role is stuck in traffic, give it 15 seconds...")
time.sleep(15)
print("Role is ready.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: DQ ruleset exists, DQDL contains exactly 12 rules across the
# six required families, TargetTable points at events_silver.

def checkpoint_1(session):
    try:
        glue_client = boto3.client(
            "glue",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            rs = glue_client.get_data_quality_ruleset(Name=RULESET_NAME)
        except ClientError as e:
            if e.response["Error"]["Code"] == "EntityNotFoundException":
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"DQ ruleset {RULESET_NAME!r} does not exist. Run Task 1."
                    ),
                )
            return CheckpointResult(status="error", error_reason=str(e))

        target = rs.get("TargetTable") or {}
        if target.get("TableName") != "events_silver":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DQ ruleset TargetTable is {target!r}; expected TableName=events_silver."
                ),
            )

        dqdl = rs.get("Ruleset", "") or ""
        # Family detection. Each pattern matches a single rule occurrence in DQDL.
        family_patterns = {
            "ColumnExists": r"\bColumnExists\b",
            "Uniqueness": r"\bIsUnique\b",
            "ColumnValues": r"\bColumnValues\b",
            "ReferentialIntegrity": r"\bReferentialIntegrity\b",
            "RowCount": r"\bRowCount\b",
            "Freshness": r"\bDataFreshness\b",
        }
        family_counts = {fam: len(re.findall(pat, dqdl)) for fam, pat in family_patterns.items()}
        total_rules = sum(family_counts.values())
        missing_families = [fam for fam, n in family_counts.items() if n == 0]
        if missing_families:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DQDL is missing rules from these families: {missing_families}. "
                    f"Counts found: {family_counts}."
                ),
            )
        if total_rules != 12:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DQDL contains {total_rules} recognized rules; expected 12. "
                    f"Counts found: {family_counts}."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the ruleset back from the Glue API and parses the DQDL string. It cares about three things: the ruleset exists, the TargetTable points at `events_silver`, and the DQDL contains exactly 12 rules spread across all six families (ColumnExists, Uniqueness via `IsUnique`, ColumnValues, ReferentialIntegrity, RowCount, Freshness via `DataFreshness`). Most early failures here are missing one family or an off-by-one on the rule count.

</details>

<details><summary>Hint 2 (direction)</summary>

DQDL syntax is `Rules = [ Rule1, Rule2, ... ]` with one keyword per family. `IsUnique` is the Uniqueness family. `DataFreshness` is the Freshness family. Severity (critical vs warning) is NOT a DQDL feature; do not try to encode it in the ruleset. Register the ruleset with `glue.create_data_quality_ruleset(Name=..., Ruleset=DQDL_STRING, TargetTable={"DatabaseName": DB, "TableName": "events_silver"})`.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the working DQDL plus the registration call:

```python
DQDL = (
    'Rules = [\n'
    '    ColumnExists "event_id",\n'
    '    ColumnExists "customer_id",\n'
    '    ColumnExists "amount",\n'
    '    ColumnExists "event_timestamp",\n'
    '    IsUnique "event_id",\n'
    '    ColumnValues "amount" between 0 and 100000,\n'
    '    ColumnValues "customer_id" matches "C[0-9]+",\n'
    '    ReferentialIntegrity "customer_id" "ref_customers.customer_id" >= 0.95,\n'
    '    RowCount > 20,\n'
    '    RowCount between 25 and 200,\n'
    '    DataFreshness "event_timestamp" <= 24 hours,\n'
    '    DataFreshness "event_timestamp" >= 0 hours\n'
    ']'
)

glue.create_data_quality_ruleset(
    Name=RULESET_NAME,
    Description="12-rule DQ ruleset for the data-quality-framework lab",
    Ruleset=DQDL,
    TargetTable={"DatabaseName": DB, "TableName": "events_silver"},
    Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
)
```

If a previous run left the ruleset behind, the orphan scan would have blocked setup. If you hit `AlreadyExistsException` here, use `glue.update_data_quality_ruleset(Name=..., Ruleset=DQDL)` to overwrite.

</details>

**Wallet check.** About 1 to 2 cents so far. The Glue catalog and IAM operations are free. Athena DDL and the seed INSERTs scan tiny amounts (kilobytes); each query is a fraction of a penny. The DQ ruleset itself bills nothing on registration; you only pay when you run an evaluation. Your morning coffee cost 200x more than this cell.

## Task 2: Build the router Lambda, the SNS topic, the state machine, and the EventBridge rule

Now you wire the routing layer. Four pieces:

1. **Router Lambda.** One function, three actions controlled by the `action` field in the event payload. `action="evaluate"` starts a Glue DQ evaluation run, polls until terminal, and returns the rule-by-rule pass/fail list. `action="route"` reads that result and emits `{severity, failing_rules}`; severity is `clean` if everything passes, `warning` if only DataFreshness failed, `critical` otherwise (ColumnExists / Uniqueness / ColumnValues / ReferentialIntegrity / RowCount failures all trip critical). `action="write_gold"` runs `INSERT INTO events_gold SELECT * FROM events_silver` via Athena. The Lambda runs as the same role you built in Task 1.
2. **SNS topic.** No subscription. The lab measures publishes via the `AWS/SNS NumberOfMessagesPublished` CloudWatch metric, so a subscriber is not required to validate the wiring. In production this is where the on-call PagerDuty webhook lives.
3. **Step Functions state machine.** ASL with the following named states: `EvaluateRuleset` (Lambda invoke, action=evaluate) → `RouterLambda` (Lambda invoke, action=route) → `SeveritySwitch` (Choice on `$.router.Payload.severity`) → `BlockGoldWrite` (Fail) for critical, `PublishWarning` (SNS publish) followed by `GoldWrite` (Lambda invoke, action=write_gold) for warning, or `GoldWrite` directly for clean.
4. **EventBridge rule.** Listens on `aws.glue` source with `detail-type="Data Quality Evaluation Results Available"` and targets the SNS topic. This is the secondary monitoring channel: even if the state machine fails before reaching `PublishWarning`, the raw Glue DQ event still surfaces to operators. The lab does not validate this rule fires (Glue DQ event emission is best-effort); the rule is in the manifest so cleanup tears it down.

The state machine state names matter. The Checkpoint 2 validator parses the ASL JSON and looks for `EvaluateRuleset`, `RouterLambda`, `SeveritySwitch`, `BlockGoldWrite`, `PublishWarning`, and `GoldWrite`. Rename a state and the validator fails.

In [ ]:
# NBVAL_SKIP
# Task 2: Router Lambda + SNS topic + Step Functions state machine + EventBridge rule.

lambda_client = boto3.client(
    "lambda",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sns = boto3.client(
    "sns",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
events = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sfn = boto3.client(
    "stepfunctions",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

# Router Lambda source. In-memory zip; no S3 staging needed.
ROUTER_SRC = '''
import json
import os
import time

import boto3

DB = os.environ["DB"]
RULESET_NAME = os.environ["RULESET_NAME"]
ROLE_ARN = os.environ["ROLE_ARN"]
ATHENA_OUTPUT_LOC = os.environ["ATHENA_OUTPUT_LOC"]


def _start_and_wait_dq():
    glue = boto3.client("glue")
    resp = glue.start_data_quality_ruleset_evaluation_run(
        DataSource={"GlueTable": {"DatabaseName": DB, "TableName": "events_silver"}},
        Role=ROLE_ARN,
        RulesetNames=[RULESET_NAME],
        AdditionalDataSources={
            "ref_customers": {"DatabaseName": DB, "TableName": "ref_customers"}
        },
    )
    run_id = resp["RunId"]
    last = {}
    deadline = time.time() + 200
    while time.time() < deadline:
        last = glue.get_data_quality_ruleset_evaluation_run(RunId=run_id)
        if last.get("Status") in ("SUCCEEDED", "FAILED", "STOPPED", "TIMEOUT"):
            break
        time.sleep(5)
    rule_results = []
    result_ids = last.get("ResultIds", []) or []
    if result_ids:
        batched = glue.batch_get_data_quality_result(ResultIds=result_ids)
        for res in batched.get("Results", []):
            for rr in res.get("RuleResults", []):
                rule_results.append({
                    "name": rr.get("Name"),
                    "description": rr.get("Description"),
                    "result": rr.get("Result"),
                    "evaluation_message": rr.get("EvaluationMessage", ""),
                })
    return {
        "run_id": run_id,
        "status": last.get("Status"),
        "rule_results": rule_results,
    }


def _classify(dq):
    rule_results = dq.get("rule_results", [])
    failures = [r for r in rule_results if (r.get("result") or "").upper() != "PASS"]
    if not failures:
        return {"severity": "clean", "failing_rules": []}
    only_freshness = True
    failing_names = []
    for r in failures:
        descriptor = ((r.get("description") or "") + " " + (r.get("name") or "")).lower()
        failing_names.append(r.get("description") or r.get("name") or "unnamed")
        if "datafreshness" not in descriptor and "freshness" not in descriptor:
            only_freshness = False
    severity = "warning" if only_freshness else "critical"
    return {"severity": severity, "failing_rules": failing_names}


def _write_gold():
    athena = boto3.client("athena")
    sql = (
        "INSERT INTO " + DB + ".events_gold "
        "SELECT event_id, customer_id, amount, event_timestamp FROM " + DB + ".events_silver"
    )
    resp = athena.start_query_execution(
        QueryString=sql,
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT_LOC},
        QueryExecutionContext={"Database": DB},
    )
    qid = resp["QueryExecutionId"]
    deadline = time.time() + 120
    state = ""
    while time.time() < deadline:
        st = athena.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        state = st["State"]
        if state in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(2)
    return {"state": state, "query_id": qid}


def handler(event, context):
    action = (event or {}).get("action", "")
    if action == "evaluate":
        return _start_and_wait_dq()
    if action == "route":
        dq = (event or {}).get("dq_result") or {}
        return _classify(dq)
    if action == "write_gold":
        return _write_gold()
    return {"error": "unknown action", "action": action}
'''.lstrip()

# Build the in-memory zip.
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr("lambda_function.py", ROUTER_SRC)
router_zip = buf.getvalue()

try:
    lambda_client.create_function(
        FunctionName=ROUTER_NAME,
        Runtime="python3.11",
        Role=ROLE_ARN,
        Handler="lambda_function.handler",
        Code={"ZipFile": router_zip},
        Timeout=300,
        MemorySize=256,
        Environment={"Variables": {
            "DB": DB,
            "RULESET_NAME": RULESET_NAME,
            "ROLE_ARN": ROLE_ARN,
            "ATHENA_OUTPUT_LOC": ATHENA_OUTPUT_LOC,
        }},
        Tags={LAB_TAG_KEY: LAB_TAG_VALUE},
    )
    print(f"Router Lambda created: {ROUTER_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceConflictException":
        lambda_client.update_function_code(FunctionName=ROUTER_NAME, ZipFile=router_zip)
        print(f"Router Lambda {ROUTER_NAME} already existed, code updated.")
    else:
        raise

# SNS topic.
try:
    sns.create_topic(
        Name=TOPIC_NAME,
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"SNS topic created: {TOPIC_NAME}")
except ClientError as e:
    print(f"SNS create_topic raised {e.response['Error']['Code']}; reusing {TOPIC_ARN}")

# Step Functions state machine. ASL definition.
ASL = {
    "Comment": "DQ-gated gold write with severity routing",
    "StartAt": "EvaluateRuleset",
    "States": {
        "EvaluateRuleset": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {
                "FunctionName": ROUTER_ARN,
                "Payload": {"action": "evaluate"},
            },
            "ResultPath": "$.dq_result",
            "Next": "RouterLambda",
        },
        "RouterLambda": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {
                "FunctionName": ROUTER_ARN,
                "Payload": {
                    "action": "route",
                    "dq_result.$": "$.dq_result.Payload",
                },
            },
            "ResultPath": "$.router",
            "Next": "SeveritySwitch",
        },
        "SeveritySwitch": {
            "Type": "Choice",
            "Choices": [
                {
                    "Variable": "$.router.Payload.severity",
                    "StringEquals": "critical",
                    "Next": "BlockGoldWrite",
                },
                {
                    "Variable": "$.router.Payload.severity",
                    "StringEquals": "warning",
                    "Next": "PublishWarning",
                },
            ],
            "Default": "GoldWrite",
        },
        "BlockGoldWrite": {
            "Type": "Fail",
            "Error": "CriticalDQViolation",
            "Cause": "A critical assertion fired; gold write blocked.",
        },
        "PublishWarning": {
            "Type": "Task",
            "Resource": "arn:aws:states:::sns:publish",
            "Parameters": {
                "TopicArn": TOPIC_ARN,
                "Subject": "DQ warning on events_silver",
                "Message": "Freshness assertion fired. Gold write proceeding because severity is warning, not critical.",
            },
            "ResultPath": "$.sns",
            "Next": "GoldWrite",
        },
        "GoldWrite": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {
                "FunctionName": ROUTER_ARN,
                "Payload": {"action": "write_gold"},
            },
            "ResultPath": "$.gold",
            "End": True,
        },
    },
}

try:
    sfn.create_state_machine(
        name=SM_NAME,
        definition=json.dumps(ASL),
        roleArn=ROLE_ARN,
        type="STANDARD",
        tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
    )
    print(f"State machine created: {SM_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "StateMachineAlreadyExists":
        sfn.update_state_machine(
            stateMachineArn=STATE_MACHINE_ARN,
            definition=json.dumps(ASL),
            roleArn=ROLE_ARN,
        )
        print(f"State machine {SM_NAME} already existed, definition updated.")
    else:
        raise

# EventBridge rule on Glue DQ events, targeting SNS. Best-effort monitoring;
# Glue DQ event emission is not used by checkpoint validation.
event_pattern = {
    "source": ["aws.glue"],
    "detail-type": ["Data Quality Evaluation Results Available"],
}
events.put_rule(
    Name=RULE_NAME,
    EventPattern=json.dumps(event_pattern),
    State="ENABLED",
    Description="Mirrors Glue DQ result events to SNS for the data-quality-framework lab",
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
events.put_targets(
    Rule=RULE_NAME,
    Targets=[{"Id": "1", "Arn": TOPIC_ARN}],
)
# SNS topic policy so EventBridge can publish.
topic_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "events.amazonaws.com"},
        "Action": "sns:Publish",
        "Resource": TOPIC_ARN,
    }],
}
sns.set_topic_attributes(
    TopicArn=TOPIC_ARN,
    AttributeName="Policy",
    AttributeValue=json.dumps(topic_policy),
)
print(f"EventBridge rule created: {RULE_NAME} (target: SNS topic {TOPIC_NAME})")

# Brief Lambda warm-up. Step Functions invokes against the new function can
# 404 for a few seconds after create_function.
print("Letting Lambda settle...")
time.sleep(8)
print("Routing layer ready.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: state machine ASL has all six expected named states.

def checkpoint_2(session):
    try:
        sfn_client = boto3.client(
            "stepfunctions",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            desc = sfn_client.describe_state_machine(stateMachineArn=STATE_MACHINE_ARN)
        except ClientError as e:
            code_str = e.response["Error"]["Code"]
            if code_str == "StateMachineDoesNotExist":
                return CheckpointResult(
                    status="fail",
                    error_reason=f"State machine {SM_NAME} does not exist. Run Task 2.",
                )
            return CheckpointResult(status="error", error_reason=str(e))
        try:
            definition = json.loads(desc.get("definition", "{}"))
        except json.JSONDecodeError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"State machine definition is not valid JSON: {e}",
            )
        states = definition.get("States", {})
        expected = {
            "EvaluateRuleset",
            "RouterLambda",
            "SeveritySwitch",
            "BlockGoldWrite",
            "PublishWarning",
            "GoldWrite",
        }
        missing = sorted(expected - set(states.keys()))
        if missing:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"State machine ASL is missing required state names: {missing}. "
                    f"Found: {sorted(states.keys())}."
                ),
            )
        # Sanity-check the Choice routes critical to BlockGoldWrite and warning to
        # PublishWarning. Default must NOT be BlockGoldWrite (that would block clean runs).
        sw = states.get("SeveritySwitch", {})
        if sw.get("Type") != "Choice":
            return CheckpointResult(
                status="fail",
                error_reason="SeveritySwitch must be a Choice state.",
            )
        choices = sw.get("Choices", [])
        crit_next = next((c.get("Next") for c in choices if c.get("StringEquals") == "critical"), None)
        warn_next = next((c.get("Next") for c in choices if c.get("StringEquals") == "warning"), None)
        if crit_next != "BlockGoldWrite":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Critical branch must transition to BlockGoldWrite; found Next={crit_next!r}."
                ),
            )
        if warn_next != "PublishWarning":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Warning branch must transition to PublishWarning; found Next={warn_next!r}."
                ),
            )
        if sw.get("Default") != "GoldWrite":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Choice Default must be GoldWrite (the clean path); found {sw.get('Default')!r}."
                ),
            )
        if states.get("BlockGoldWrite", {}).get("Type") != "Fail":
            return CheckpointResult(
                status="fail",
                error_reason="BlockGoldWrite must be a Fail state.",
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the state machine definition back from the Step Functions API and parses it as JSON. It checks for six named states (`EvaluateRuleset`, `RouterLambda`, `SeveritySwitch`, `BlockGoldWrite`, `PublishWarning`, `GoldWrite`) and verifies that the Choice routes `critical` to the Fail state and `warning` to the SNS publish state, with `clean` falling through the default to `GoldWrite`. A mis-named state (e.g., `EvaluateDQ` instead of `EvaluateRuleset`) is the most common failure.

</details>

<details><summary>Hint 2 (direction)</summary>

Use the AWS service integration ARNs in the ASL Resource fields: `arn:aws:states:::lambda:invoke` for the three Lambda calls, `arn:aws:states:::sns:publish` for the warning publish. The Choice state's `Variable` reads `$.router.Payload.severity` because Lambda invoke wraps the function's return value in a Payload object. The Fail state needs `Error` and `Cause` (no `Next`, no `End`). The Default of the Choice must be `GoldWrite` so the clean path runs without a critical or warning match.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the working ASL plus the create call:

```python
ASL = {
    "Comment": "DQ-gated gold write with severity routing",
    "StartAt": "EvaluateRuleset",
    "States": {
        "EvaluateRuleset": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {"FunctionName": ROUTER_ARN, "Payload": {"action": "evaluate"}},
            "ResultPath": "$.dq_result",
            "Next": "RouterLambda",
        },
        "RouterLambda": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {
                "FunctionName": ROUTER_ARN,
                "Payload": {"action": "route", "dq_result.$": "$.dq_result.Payload"},
            },
            "ResultPath": "$.router",
            "Next": "SeveritySwitch",
        },
        "SeveritySwitch": {
            "Type": "Choice",
            "Choices": [
                {"Variable": "$.router.Payload.severity", "StringEquals": "critical", "Next": "BlockGoldWrite"},
                {"Variable": "$.router.Payload.severity", "StringEquals": "warning", "Next": "PublishWarning"},
            ],
            "Default": "GoldWrite",
        },
        "BlockGoldWrite": {
            "Type": "Fail",
            "Error": "CriticalDQViolation",
            "Cause": "A critical assertion fired; gold write blocked.",
        },
        "PublishWarning": {
            "Type": "Task",
            "Resource": "arn:aws:states:::sns:publish",
            "Parameters": {
                "TopicArn": TOPIC_ARN,
                "Subject": "DQ warning on events_silver",
                "Message": "Freshness assertion fired. Gold write proceeding because severity is warning, not critical.",
            },
            "ResultPath": "$.sns",
            "Next": "GoldWrite",
        },
        "GoldWrite": {
            "Type": "Task",
            "Resource": "arn:aws:states:::lambda:invoke",
            "Parameters": {"FunctionName": ROUTER_ARN, "Payload": {"action": "write_gold"}},
            "ResultPath": "$.gold",
            "End": True,
        },
    },
}

sfn.create_state_machine(
    name=SM_NAME,
    definition=json.dumps(ASL),
    roleArn=ROLE_ARN,
    type="STANDARD",
    tags=[{"key": LAB_TAG_KEY, "value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Still under 5 cents. Lambda create + SNS create + EventBridge rule create + Step Functions create are all free control-plane calls. The router Lambda has not been invoked yet (Task 3 is the first real invocation); no compute charges accrued. Your morning coffee still beats this lab by two orders of magnitude.

## Task 3: Clean run

Now you trigger the first state machine execution. Silver already holds the 40-row clean dataset from Task 1, so the DQ evaluation should pass all 12 assertions. The router should classify the run as `clean`. The Choice should fall through to its Default (`GoldWrite`). Gold should receive 40 new rows. SNS should NOT publish.

The validator checks four things:
1. The state machine execution reaches `SUCCEEDED`.
2. `events_gold` row count is at least 40 (the rows that just flowed through).
3. The router payload reports `severity="clean"`.
4. `AWS/SNS NumberOfMessagesPublished` for the lab topic in the last 5 minutes is 0.

The poll loop has a 5-minute ceiling. Glue DQ on a 40-row Iceberg table typically lands in 60-90 seconds.

Helper functions for execution polling and gold counting are defined here once and reused by Tasks 3, 4, and 5.

In [ ]:
# NBVAL_SKIP
# Task 3: Trigger the clean run.

# Helpers used by Tasks 3, 4, 5.
def start_execution_and_wait(label: str, timeout: int = 360) -> dict:
    """Start one state machine execution and poll until it terminates."""
    sfn_client = boto3.client(
        "stepfunctions",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    start_resp = sfn_client.start_execution(
        stateMachineArn=STATE_MACHINE_ARN,
        name=f"labrun-{label}-{int(time.time())}",
        input=json.dumps({"label": label}),
    )
    exec_arn = start_resp["executionArn"]
    print(f"Started execution for {label}: {exec_arn.split(':')[-1]}")
    # Step Functions is choosing your fate. Polling for terminal state...
    deadline = time.time() + timeout
    while time.time() < deadline:
        desc = sfn_client.describe_execution(executionArn=exec_arn)
        status = desc["status"]
        if status in ("SUCCEEDED", "FAILED", "TIMED_OUT", "ABORTED"):
            return {"status": status, "arn": exec_arn, "output": desc.get("output", "")}
        time.sleep(6)
    return {"status": "TIMEOUT_LOCAL", "arn": exec_arn, "output": ""}


def gold_row_count() -> int:
    athena_client = boto3.client(
        "athena",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    resp = athena_client.start_query_execution(
        QueryString=f"SELECT COUNT(*) AS n FROM {DB}.events_gold",
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT_LOC},
        QueryExecutionContext={"Database": DB},
    )
    qid = resp["QueryExecutionId"]
    deadline = time.time() + 60
    while time.time() < deadline:
        st = athena_client.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]["State"]
        if st in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(2)
    if st != "SUCCEEDED":
        return -1
    results = athena_client.get_query_results(QueryExecutionId=qid)
    rows = results.get("ResultSet", {}).get("Rows", [])
    # Row 0 is the header; row 1 is the count value.
    if len(rows) < 2:
        return 0
    val = rows[1]["Data"][0].get("VarCharValue", "0")
    try:
        return int(val)
    except (TypeError, ValueError):
        return 0


def sns_publishes_last_5_minutes() -> int:
    cw_client = boto3.client(
        "cloudwatch",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    end_time = datetime.now(timezone.utc)
    start_time = end_time - timedelta(minutes=5)
    stats = cw_client.get_metric_statistics(
        Namespace="AWS/SNS",
        MetricName="NumberOfMessagesPublished",
        Dimensions=[{"Name": "TopicName", "Value": TOPIC_NAME}],
        StartTime=start_time,
        EndTime=end_time,
        Period=60,
        Statistics=["Sum"],
    )
    return int(sum(dp.get("Sum", 0) for dp in stats.get("Datapoints", [])))


# Glue DQ is running 12 assertions. About 60 to 90 seconds...
print("Glue DQ is running 12 assertions. About 60 to 90 seconds...")
# YOUR CODE: Call start_execution_and_wait("clean") and store the result in clean_run.
print(f"Clean execution status: {clean_run['status']}")
print(f"events_gold row count after clean run: {gold_row_count()}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: clean run succeeded, gold has >= 40 rows, SNS metric shows 0
# publishes in the last 5 minutes.

def checkpoint_3(session):
    try:
        # The state machine's most-recent execution must be SUCCEEDED with
        # the router payload severity = clean.
        sfn_client = boto3.client(
            "stepfunctions",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        execs = sfn_client.list_executions(
            stateMachineArn=STATE_MACHINE_ARN, maxResults=1
        )
        items = execs.get("executions", [])
        if not items:
            return CheckpointResult(
                status="fail",
                error_reason="No state machine executions found. Run Task 3.",
            )
        latest = items[0]
        if latest["status"] != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest execution status is {latest['status']!r}, expected SUCCEEDED. "
                    f"Check the execution's failure details in the Step Functions console."
                ),
            )
        desc = sfn_client.describe_execution(executionArn=latest["executionArn"])
        output = json.loads(desc.get("output", "{}") or "{}")
        severity = (
            output.get("router", {})
            .get("Payload", {})
            .get("severity")
        )
        if severity != "clean":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest execution router severity is {severity!r}, expected 'clean'. "
                    f"Confirm silver is the 40-row fresh dataset and re-run."
                ),
            )
        # Gold row count.
        count = gold_row_count()
        if count < 40:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"events_gold has {count} rows after the clean run; expected at least 40. "
                    f"Did the GoldWrite state run to completion?"
                ),
            )
        # SNS must not have published during the clean run.
        publishes = sns_publishes_last_5_minutes()
        if publishes > 0:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AWS/SNS NumberOfMessagesPublished for {TOPIC_NAME} shows {publishes} "
                    f"publish(es) in the last 5 minutes; expected 0 for a clean run."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the most recent execution off the state machine and inspects the output. It needs the execution status to be `SUCCEEDED`, the gold row count to be at least 40, and zero SNS publishes in the last 5 minutes. If the cell ran but no execution was started, the validator will say no executions found.

</details>

<details><summary>Hint 2 (direction)</summary>

Call `start_execution_and_wait("clean")` from this cell. The helper handles start, polling, and terminal state detection. Assign the return value to `clean_run` so the print statement after it can show the execution status. If the execution fails on the EvaluateRuleset state, the most common cause is silver not actually being the 40-row clean dataset (re-run the seed code from Task 1 if you have been experimenting).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the full working cell content:

```python
print("Glue DQ is running 12 assertions. About 60 to 90 seconds...")
clean_run = start_execution_and_wait("clean")
print(f"Clean execution status: {clean_run['status']}")
print(f"events_gold row count after clean run: {gold_row_count()}")
```

</details>

**Wallet check.** Running total: about 10 to 15 cents. The Glue DQ evaluation on a 40-row Iceberg table bills about 0.1 cent (well under the $0.10 per million records pricing). The Athena gold INSERT scans kilobytes; another fraction of a penny. Step Functions Standard charges $0.025 per 1,000 transitions and this execution used six. The CloudWatch metric query is free. Your coffee is still winning by an order of magnitude.

## Task 4: Warning run (stale silver)

Replace silver with the same 40 rows but with `event_timestamp` set 25 hours ago. Everything else is identical: same schema, same row count, same customer IDs. The DataFreshness rule `DataFreshness "event_timestamp" <= 24 hours` fires. No other rules fail.

The router classifies the run as `warning`. The Choice routes through `PublishWarning` (SNS publish) then `GoldWrite`. Gold receives another 40 rows. SNS shows at least 1 publish.

The validator checks:
1. Latest execution is `SUCCEEDED` (warning does not block).
2. Router severity is `warning`.
3. Gold row count is at least 80 (40 from clean + 40 from this run).
4. `AWS/SNS NumberOfMessagesPublished` for the lab topic shows at least 1 publish in the last 5 minutes.

CloudWatch metric publish lag is a real thing. After the execution reaches SUCCEEDED, the SNS metric can take 60 to 90 seconds to surface. The helper sleeps 90 seconds before the checkpoint runs.

In [ ]:
# NBVAL_SKIP
# Task 4: Replace silver with a 25-hour-stale dataset, trigger execution.

def silver_stale_sql() -> str:
    rows = []
    for i in range(1, 41):
        cid = f"C00{((i - 1) % 5) + 1}"
        amt = round(10 + (i * 3.7) % 200, 2)
        # 25 hours ago: trips DataFreshness <= 24 hours.
        rows.append(
            f"('E{i:04d}', '{cid}', {amt}, current_timestamp - interval '25' hour)"
        )
    values = ", ".join(rows)
    return (
        f"INSERT INTO {DB}.events_silver (event_id, customer_id, amount, event_timestamp) "
        f"VALUES {values}"
    )

# Reset silver to 40 stale rows. DELETE then INSERT for the Iceberg swap.
# YOUR CODE: Reset silver to the stale dataset. Two run_athena calls:
# first DELETE FROM events_silver, then the silver_stale_sql() INSERT.
# Each call returns a dict; raise RuntimeError if state != "SUCCEEDED".

print("Glue DQ is running 12 assertions. About 60 to 90 seconds...")
warning_run = start_execution_and_wait("warning")
print(f"Warning execution status: {warning_run['status']}")
print(f"events_gold row count after warning run: {gold_row_count()}")
# CloudWatch metric publish lag is a real thing. Sleeping 90 seconds...
print("CloudWatch metric publish lag is a real thing. Sleeping 90 seconds...")
time.sleep(90)
print(f"SNS publishes in the last 5 minutes: {sns_publishes_last_5_minutes()}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: warning run succeeded, severity=warning, gold >= 80 rows,
# SNS metric shows >= 1 publish.

def checkpoint_4(session):
    try:
        sfn_client = boto3.client(
            "stepfunctions",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        execs = sfn_client.list_executions(
            stateMachineArn=STATE_MACHINE_ARN, maxResults=1
        )
        items = execs.get("executions", [])
        if not items:
            return CheckpointResult(
                status="fail",
                error_reason="No state machine executions found. Run Task 4.",
            )
        latest = items[0]
        if latest["status"] != "SUCCEEDED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest execution status is {latest['status']!r}, expected SUCCEEDED. "
                    f"Warning severity must not block; check the router's severity classification."
                ),
            )
        desc = sfn_client.describe_execution(executionArn=latest["executionArn"])
        output = json.loads(desc.get("output", "{}") or "{}")
        severity = (
            output.get("router", {})
            .get("Payload", {})
            .get("severity")
        )
        if severity != "warning":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest execution router severity is {severity!r}, expected 'warning'. "
                    f"Confirm silver was reseeded with 25-hour-stale timestamps."
                ),
            )
        count = gold_row_count()
        if count < 80:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"events_gold has {count} rows after the warning run; expected at least 80 "
                    f"(40 from clean + 40 from warning). GoldWrite may not have completed."
                ),
            )
        publishes = sns_publishes_last_5_minutes()
        if publishes < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AWS/SNS NumberOfMessagesPublished for {TOPIC_NAME} shows {publishes} "
                    f"publishes in the last 5 minutes; expected at least 1. CloudWatch metric "
                    f"emission can lag 60-90 seconds; wait and re-run, or confirm PublishWarning "
                    f"actually executed (check Step Functions execution history)."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint runs four sub-checks: execution SUCCEEDED, router severity is `warning`, gold has at least 80 rows total, and SNS published at least once. The most common failure here is that silver was not actually replaced with stale data before the execution started, so the run reports `severity="clean"` and SNS never publishes.

</details>

<details><summary>Hint 2 (direction)</summary>

Athena Iceberg supports `DELETE FROM <table>` followed by `INSERT INTO <table> VALUES ...`. The `silver_stale_sql()` helper already builds the 25-hour-stale INSERT for you. You need to call `run_athena(f"DELETE FROM {DB}.events_silver")` first, check that the state is `SUCCEEDED`, then call `run_athena(silver_stale_sql())` and check the same. Raise an exception if either DDL fails; the state machine evaluating against an empty or half-populated silver table is a confusing failure mode.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the full working solution for the silver swap:

```python
del_result = run_athena(f"DELETE FROM {DB}.events_silver")
if del_result["state"] != "SUCCEEDED":
    raise RuntimeError(f"stale silver DELETE failed: {del_result}")
ins_result = run_athena(silver_stale_sql())
if ins_result["state"] != "SUCCEEDED":
    raise RuntimeError(f"stale silver INSERT failed: {ins_result}")
print("events_silver replaced with 40 stale rows (event_timestamp -25h).")
```

</details>

**Wallet check.** Running total: about 20 to 30 cents. Two Glue DQ evaluations now and two Athena gold writes. Step Functions has consumed about a dozen transitions across two executions; still pennies. The SNS publish itself costs nothing for the first 1 million publishes per month. Your coffee is starting to look reasonable in comparison.

## Task 5: Critical run (row-count drop)

Replace silver with 18 rows. The 18 rows have fresh timestamps (no freshness violation) and valid customer IDs (no ColumnValues or ReferentialIntegrity violation). The only thing that fails is `RowCount > 20` and `RowCount between 25 and 200`. Two RowCount rules in the critical family fire.

The router classifies the run as `critical` (any failure outside the freshness family is critical). The Choice routes to `BlockGoldWrite` (Fail state). The execution ends in `FAILED`. The gold write does NOT happen; row count stays at 80 from the warning run. SNS publishes at least once because the EventBridge rule on Glue DQ result events still fires (the state machine's PublishWarning state did not run, but Glue DQ emitted a result event regardless).

Wait. The EventBridge rule is set up to fire on Glue DQ result events but those are emitted regardless of pass/fail. Both clean and warning runs would also fire the EB rule, but the checkpoints for clean/warning rely on the state machine's own SNS publish, not the EB-driven one. For the critical run the validator looks at the 5-minute SNS metric window and requires at least one publish, which the EB rule provides.

The validator checks:
1. Latest execution status is `FAILED` (BlockGoldWrite is a Fail state).
2. Gold row count is UNCHANGED from the warning run (still at least 80, and not larger than 80 + any seed buffer).
3. SNS metric shows at least 1 publish in the last 5 minutes (from the EventBridge rule on the Glue DQ event).

In [ ]:
# NBVAL_SKIP
# Task 5: Replace silver with 18 rows (>30% drop), trigger execution.

def silver_rowdrop_sql() -> str:
    rows = []
    for i in range(1, 19):
        cid = f"C00{((i - 1) % 5) + 1}"
        amt = round(10 + (i * 3.7) % 200, 2)
        rows.append(f"('E{i:04d}', '{cid}', {amt}, current_timestamp)")
    values = ", ".join(rows)
    return (
        f"INSERT INTO {DB}.events_silver (event_id, customer_id, amount, event_timestamp) "
        f"VALUES {values}"
    )

gold_count_before_critical = gold_row_count()
print(f"events_gold row count BEFORE critical run: {gold_count_before_critical}")

# YOUR CODE: Reset silver to the row-count-drop dataset. Two run_athena calls:
# first DELETE FROM events_silver, then the silver_rowdrop_sql() INSERT.
# Each call returns a dict; raise RuntimeError if state != "SUCCEEDED".

print("Step Functions is choosing your fate. Polling for terminal state...")
critical_run = start_execution_and_wait("critical")
print(f"Critical execution status: {critical_run['status']}")
gold_count_after_critical = gold_row_count()
print(f"events_gold row count AFTER critical run: {gold_count_after_critical}")
print("CloudWatch metric publish lag is a real thing. Sleeping 90 seconds...")
time.sleep(90)
print(f"SNS publishes in the last 5 minutes: {sns_publishes_last_5_minutes()}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: critical run ended FAILED, gold count unchanged, SNS >= 1.

def checkpoint_5(session):
    try:
        sfn_client = boto3.client(
            "stepfunctions",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        execs = sfn_client.list_executions(
            stateMachineArn=STATE_MACHINE_ARN, maxResults=1
        )
        items = execs.get("executions", [])
        if not items:
            return CheckpointResult(
                status="fail",
                error_reason="No state machine executions found. Run Task 5.",
            )
        latest = items[0]
        if latest["status"] != "FAILED":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Latest execution status is {latest['status']!r}, expected FAILED. "
                    f"BlockGoldWrite must be a Fail state and the critical branch must reach it."
                ),
            )
        count = gold_row_count()
        # Allow a small forgiving margin in case the student re-ran the warning
        # cell before reaching here. The critical run MUST NOT write gold; the
        # count after critical must equal the count before critical.
        if count != gold_count_before_critical:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"events_gold row count is {count} after the critical run; "
                    f"expected exactly {gold_count_before_critical} (unchanged from before). "
                    f"If this is larger, the GoldWrite state ran when it should have been blocked."
                ),
            )
        publishes = sns_publishes_last_5_minutes()
        if publishes < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"AWS/SNS NumberOfMessagesPublished for {TOPIC_NAME} shows {publishes} "
                    f"publishes in the last 5 minutes; expected at least 1 from the "
                    f"EventBridge rule on the Glue DQ result event. Verify the EventBridge "
                    f"rule target is the lab SNS topic and the topic policy allows "
                    f"events.amazonaws.com to publish."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

<details><summary>Hint 1 (nudge)</summary>

The checkpoint reads the most recent execution and expects status `FAILED` (because `BlockGoldWrite` is a Fail state). It also expects the gold row count to be exactly what it was before the critical run. If gold grew, the state machine wrote gold even though severity was critical, which means the Choice routing or the BlockGoldWrite state is wrong.

</details>

<details><summary>Hint 2 (direction)</summary>

Same DELETE-then-INSERT pattern as Task 4, but with the `silver_rowdrop_sql()` helper that emits 18 rows instead of 40. With `RowCount > 20` and `RowCount between 25 and 200` in the ruleset, 18 rows fails both. The router classifies any non-freshness failure as critical, so the Choice routes to `BlockGoldWrite` and the execution ends FAILED. If your execution lands SUCCEEDED instead, either silver did not actually get truncated (so the DQ run saw the old data) or the router's classification logic is wrong.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Here is the full working solution for the silver swap:

```python
del_result = run_athena(f"DELETE FROM {DB}.events_silver")
if del_result["state"] != "SUCCEEDED":
    raise RuntimeError(f"row-drop silver DELETE failed: {del_result}")
ins_result = run_athena(silver_rowdrop_sql())
if ins_result["state"] != "SUCCEEDED":
    raise RuntimeError(f"row-drop silver INSERT failed: {ins_result}")
print("events_silver replaced with 18 rows (>30% row-count drop).")
```

</details>

**Wallet check.** Running total: about 30 to 50 cents. Three Glue DQ evaluations (3 x ~0.1 cent), two Athena gold writes (the critical run skipped the gold write because the state machine blocked it), about a dozen Athena DELETE / INSERT / SELECT calls on tiny Iceberg tables, and a handful of Step Functions transitions. The critical run not writing gold saved you the most expensive line item, which is exactly the point of the gate.

In [ ]:
# NBVAL_SKIP
# Cleanup. Tear down every resource in CLEANUP_MANIFEST in reverse-creation
# order. No critical (hourly-billed) resources.
#
# Step Functions delete-state-machine drops execution history. Before delete,
# stop any in-flight executions (per lab brief's risk areas section).
#
# The default AwsCleanupAdapter does not ship handlers for sfn_state_machine,
# glue_dq_ruleset, or iceberg_table. The inline _LabAdapter below adds them
# and delegates everything else to AwsCleanupAdapter unchanged.

import sys
from labrun_checks.adapters.aws import AwsCleanupAdapter


# 1. Stop any running Step Functions executions before delete-state-machine.
#    delete-state-machine succeeds with running executions but they continue
#    silently, which is exactly the kind of orphaned compute we are trying to
#    prevent. Defensive stop loop with a short timeout.
print("Stopping any in-flight state machine executions...")
sfn_stop_client = boto3.client(
    "stepfunctions",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    running = sfn_stop_client.list_executions(
        stateMachineArn=STATE_MACHINE_ARN,
        statusFilter="RUNNING",
        maxResults=50,
    )
    for ex in running.get("executions", []):
        try:
            sfn_stop_client.stop_execution(executionArn=ex["executionArn"], error="LabCleanup", cause="Lab teardown")
            print(f"  Stopped: {ex['executionArn'].split(':')[-1]}")
        except ClientError as e:
            print(f"  stop_execution raised {e.response['Error']['Code']}; continuing")
except ClientError as e:
    if e.response["Error"]["Code"] != "StateMachineDoesNotExist":
        print(f"  list_executions raised {e.response['Error']['Code']}; continuing")


class _LabAdapter:
    """AwsCleanupAdapter wrapper that adds the three types this lab needs.

    Inlined types:
      - sfn_state_machine: stepfunctions.delete_state_machine(stateMachineArn=id)
      - glue_dq_ruleset: glue.delete_data_quality_ruleset(Name=id)
      - iceberg_table: glue.delete_table(DatabaseName=parent, Name=id). The
        underlying S3 prefix is reaped by the s3_bucket cleanup at the end.

    Everything else delegates to AwsCleanupAdapter unchanged.
    """

    def __init__(self) -> None:
        self._aws = AwsCleanupAdapter()

    def _client(self, service: str, credentials: dict):
        return boto3.client(
            service,
            aws_access_key_id=credentials["aws_access_key_id"],
            aws_secret_access_key=credentials["aws_secret_access_key"],
            aws_session_token=credentials.get("aws_session_token"),
            region_name=credentials.get("region", REGION),
        )

    def delete_resource(self, credentials: dict, resource) -> None:
        if resource.type == "sfn_state_machine":
            client = self._client("stepfunctions", credentials)
            try:
                client.delete_state_machine(stateMachineArn=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "StateMachineDoesNotExist":
                    raise
            return
        if resource.type == "glue_dq_ruleset":
            client = self._client("glue", credentials)
            try:
                client.delete_data_quality_ruleset(Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return
        if resource.type == "iceberg_table":
            client = self._client("glue", credentials)
            try:
                client.delete_table(DatabaseName=resource.parent, Name=resource.id)
            except ClientError as e:
                if e.response["Error"]["Code"] != "EntityNotFoundException":
                    raise
            return
        return self._aws.delete_resource(credentials, resource)

    def describe_resource(self, credentials: dict, resource) -> bool:
        if resource.type == "sfn_state_machine":
            client = self._client("stepfunctions", credentials)
            try:
                client.describe_state_machine(stateMachineArn=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "StateMachineDoesNotExist":
                    return False
                raise
        if resource.type == "glue_dq_ruleset":
            client = self._client("glue", credentials)
            try:
                client.get_data_quality_ruleset(Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        if resource.type == "iceberg_table":
            client = self._client("glue", credentials)
            try:
                client.get_table(DatabaseName=resource.parent, Name=resource.id)
                return True
            except ClientError as e:
                if e.response["Error"]["Code"] == "EntityNotFoundException":
                    return False
                raise
        return self._aws.describe_resource(credentials, resource)

    def tag_scan(self, credentials: dict, lab_slug: str, region: str) -> list[str]:
        return self._aws.tag_scan(credentials, lab_slug, region)


# 2. Empty the S3 bucket before run_cleanup tears it down. Iceberg writes,
#    Athena results, and intermediate files accumulate; the s3_bucket deletion
#    fails with BucketNotEmpty unless we drain first.
print("Emptying the S3 bucket before teardown...")
s3_clean = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    paginator = s3_clean.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=BUCKET_NAME):
        keys = [{"Key": obj["Key"]} for obj in page.get("Contents", [])]
        if keys:
            s3_clean.delete_objects(Bucket=BUCKET_NAME, Delete={"Objects": keys})
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchBucket":
        print(f"Bucket emptying ran into: {e}. Continuing.")

# 3. Detach EventBridge targets before delete_rule.
events_clean = boto3.client(
    "events",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    events_clean.remove_targets(Rule=RULE_NAME, Ids=["1"], Force=True)
except ClientError as e:
    if e.response["Error"]["Code"] not in ("ResourceNotFoundException", "InternalException"):
        print(f"remove_targets ran into: {e}. Continuing.")

# 4. Drop inline policies on the role so iam_role delete is not blocked.
iam_clean = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
try:
    listed = iam_clean.list_role_policies(RoleName=DQ_ROLE)
    for policy_name in listed.get("PolicyNames", []):
        iam_clean.delete_role_policy(RoleName=DQ_ROLE, PolicyName=policy_name)
except ClientError as e:
    if e.response["Error"]["Code"] != "NoSuchEntity":
        print(f"Inline policy detach for {DQ_ROLE} ran into: {e}. Continuing.")

result = run_cleanup(CLEANUP_MANIFEST, adapter=_LabAdapter())

for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)
if result.warnings or result.orphans:
    print()

failed_ids: set[str] = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 0
standard_total = len(CLEANUP_MANIFEST)
critical_destroyed = critical_total
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed}")
print(f"Standard resources destroyed: {standard_destroyed}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

cleanup(status=result.status)

if failed_count > 0:
    sys.exit(1)

**Session total: $0.40 to $1.00.** Glue DQ evaluations + Athena Iceberg writes are the bulk of the spend. Step Functions, Lambda, SNS, and EventBridge are essentially free at lab volume. The cleanup cell above destroyed every resource so nothing is still accruing. If you saw zero warnings or orphans in the cleanup output, your bill should match this estimate within a couple cents.

## These are not graded. They are for you.

Three questions worth sitting with before the interview.

1. Argue for or against the severity split you implemented. Freshness as a warning, row-count drops as critical. Where would you draw the line differently if your gold table fed a customer-facing API instead of the CFO's board deck? What changes if the gold table feeds a regulatory report due in two hours and freshness is suddenly a hard constraint?

2. The DQDL ruleset and the router Lambda's severity classification live in two different places. A future engineer who adds a new DQDL rule (say, a Sum aggregate check) has to also update the router's classification logic, or the new rule's failure will default to critical. Is that the right default? Walk through two designs: encoding severity inside DQDL rule names (e.g., a naming convention like `critical_*` / `warning_*`) versus keeping severity in the routing layer as you built it. Which one survives a team of four data engineers maintaining the ruleset over a year?

3. The state machine you built handles three terminal paths in one execution. What happens if the GoldWrite Lambda task times out or errors mid-write? The execution lands FAILED but the warning SNS publish already fired and the gold table has a partial write. Sketch the compensation logic you would add (Athena Iceberg has rollback via snapshot identifiers) and where you would put it in the ASL.

## Exam-Style Practice

**Q1.** Your gold load fails a critical row-count assertion (today's load is 55% smaller than yesterday's baseline). The on-call engineer is debating whether to push the data through anyway because the dashboard refresh deadline is in 20 minutes. What is the right call from a data-quality framework perspective?

A) Bypass the gate with a manual approval. The dashboard SLA is the higher priority and a 20-minute delay is unacceptable.

B) Block the gold write, page the data team, and investigate root cause before pushing. A partial load to a CFO-facing table costs more in trust than a missed deadline.

C) Lower the row-count threshold from 30% drop to 60% drop so today's load passes, and roll the threshold back tomorrow.

D) Disable the row-count rule for this run only and re-enable it after the load lands.

<details><summary>Show answer</summary>

**B**.

- A) Wrong because the entire point of a critical gate is that it cannot be bypassed for a deadline. The blast radius of a wrong CFO number is days of explanation; the blast radius of a 20-minute delay is one Slack message.
- B) Right because this is the exact contract you built into the framework. Critical violations block the gold write, page on-call, and force a root-cause investigation before retry. The dashboard team renders yesterday's data with a freshness banner.
- C) Wrong because tuning the threshold to pass a known-bad load is how teams end up with a rule that nobody trusts. The threshold has to stay defensible across runs.
- D) Wrong for the same reason as C, with the added problem that disabling a rule for a single run is a manual operation with no audit trail.

</details>

**Q2.** You are extending the lab's pattern to a new domain: order events from a partner system that arrives daily at 3am. The partner occasionally ships data 6-12 hours late. Freshness violations are common and the on-call engineers have started ignoring the warning pages. What is the right adjustment to the framework?

A) Raise the DataFreshness threshold from 24 hours to 48 hours so the rule stops firing during late deliveries.

B) Promote the DataFreshness rule from warning to critical so the gold write blocks on stale partner data.

C) Keep the rule unchanged but reroute warning pages to a dedicated channel that does not page the on-call engineer; surface them on a dashboard reviewed at the morning standup.

D) Remove the DataFreshness rule from the ruleset entirely because the data is always stale.

<details><summary>Show answer</summary>

**C**.

- A) Tempting but wrong. Raising the threshold to 48 hours masks the actual freshness slippage and pushes the team toward an even later acceptable delivery time over the next quarter. Alert fatigue does not come from the threshold; it comes from the channel the alert lands in.
- B) Wrong because freshness on partner data is not the same kind of failure as a row-count drop. The data is correct; it is just late. Blocking the gold load on late-but-correct data is the wrong reaction.
- C) Right because the framework's severity is the routing layer, not the rule itself. Move warnings off the paging channel to a low-noise surface. The rule keeps firing; the routing adapts to the data domain.
- D) Wrong because removing the rule means a 5-day-stale delivery would not trigger anything. The framework needs the signal even if the response is not a page.

</details>

**Q3.** A new engineer joins your team and asks why severity is implemented in the router Lambda instead of as a DQDL feature. What is the most accurate explanation of the design choice?

A) Glue DQ does not have an official severity feature in DQDL, but the lab could have implemented it via DQDL rule naming conventions and a regex in the router.

B) Severity is a routing-layer decision, not a quality-assertion decision. The same rule ("row count dropped 30%") can be warning in a dev environment and critical in prod; encoding severity in DQDL would require duplicate rulesets per environment.

C) Severity is implemented in the router because Lambda is cheaper than Glue, not for any design reason.

D) DQDL supports severity via the `Severity = ...` block but the lab opted to ignore it to demonstrate the routing pattern.

<details><summary>Show answer</summary>

**B**.

- A) Half-right. DQDL has no severity feature today, but the deeper reason for the routing-layer choice is not the absence of a DQDL feature; it is that severity legitimately depends on context outside the rule body.
- B) Right because severity is contextual to who consumes the data, not intrinsic to the assertion. The router can read the deployment environment, the time of day, or the consuming team and route accordingly. DQDL is the wrong layer for any of that.
- C) Wrong because cost is not the design reason. Even if Glue DQ added a severity feature tomorrow, the router-layer pattern would still be the right call for the contextual-routing reason above.
- D) Wrong because no `Severity = ...` block exists in DQDL today. If a future Glue release adds one, the routing-layer pattern still remains more flexible.

</details>